In [17]:
!pip install pandas numpy scikit-learn matplotlib seaborn


In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import datetime


In [19]:
#Common Preprocessing Function
def preprocess_dataset(df, label_col):
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    X = df.drop(label_col, axis=1)
    y = df[label_col]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, y

In [20]:
#USB Threat Detection
def usb_detection():
    print("🧩 USB Malware Detection (CLaMP Dataset)")
    
    df = pd.read_csv("data/usb/ClaMP_Integrated-5184.csv")

    # Convert to binary labels: Benign = 0, Malicious = 1
    df['Malicious'] = df['Malware_Class'].apply(lambda x: 0 if str(x).lower() == 'benign' else 1)
    df = df.drop(columns=['Malware_Class'], errors='ignore')

    # Select only numeric columns
    df = df.select_dtypes(include=[np.number]).join(df['Malicious'])
    X, y = preprocess_dataset(df, 'Malicious')

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = GaussianNB()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    print(f"✅ USB Detection Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title("USB Malware Detection – Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()
    
    return acc

In [21]:
#Network Intrusion Detection
def network_detection():
    print("🌐 Network Intrusion Detection (UNSW-NB15 Dataset)")
    
    train = pd.read_csv("data/network/UNSW_NB15_training-set.csv")
    test = pd.read_csv("data/network/UNSW_NB15_testing-set.csv")
    df = pd.concat([train, test], ignore_index=True)

    # Drop irrelevant or ID columns
    df = df.drop(columns=['id', 'srcip', 'sport', 'dstip', 'dsport', 'attack_cat'], errors='ignore')

    # Encode categorical columns
    for col in df.select_dtypes(include=['object']).columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])

    X, y = preprocess_dataset(df, 'label')

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = GaussianNB()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    print(f"✅ Network Detection Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Reds')
    plt.title("Network Intrusion Detection – Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

    return acc

In [22]:
#Logging Function
def log_event(module, accuracy):
    with open("sentinelguard_log.csv", "a") as f:
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        f.write(f"{timestamp},{module},{accuracy:.4f}\n")
    print(f"🪪 {module} logged → Accuracy: {accuracy:.4f}")


In [16]:
pd.read_csv("sentinelguard_log.csv", names=["Timestamp","Module","Status","Accuracy"])


,Timestamp,Module,Status,Accuracy
0,2025-10-16 12:02:37,USB Module,Completed,0.0
1,2025-10-16 12:02:37,Network Module,Completed,0.0
2,2025-10-16 12:12:51,USB Module,Completed,0.0
3,2025-10-16 12:12:53,Network Module,Completed,0.0
4,2025-10-16 12:13:22,USB Module,Completed,0.0
5,2025-10-16 12:13:26,Network Module,Completed,0.0
6,2025-10-16 13:04:41,USB Module,Completed,0.0
7,2025-10-16 13:04:45,Network Module,Completed,0.0
8,2025-10-16 15:38:41,USB Module,Completed,0.0
9,2025-10-16 15:40:24,Network Module,Completed,0.0


In [ ]:
#Run the Whole Phase 1
print("🚀 Sentinel Guard – Phase 1 Execution Started")

usb_acc = usb_detection()
log_event("USB Malware Detection (CLaMP)", usb_acc)

net_acc = network_detection()
log_event("Network Intrusion Detection (UNSW-NB15)", net_acc)

print("\n✅ Phase 1 completed – results saved in sentinelguard_log.csv")
